# Phase 2D-C | Event-Locked Optogenetic Response Verification & Dataset Expansion

**Dataset:** [DANDI 000005, published version 0.220126.1853](https://dandiarchive.org/dandiset/000005/0.220126.1853)  
**Goal:** Verify intracellular voltage responses near recorded optical events and audit *five independent mice*, including two new candidates.

**What this is:** Data-quality and feasibility work with **real mouse somatosensory recordings**. It **does not** train AI, prove optical causality, demonstrate human epilepsy control, or claim clock synchronization based on overlapping timestamp ranges alone.

**Outputs:** Event-level voltage and candidate-spike summaries, time-axis diagnostics, optional circular-time-shift negative control, stimulation-protocol table, per-animal voltage plots, label-quality verdict, and `phase2dc_response_verification_results.zip`.

**Scientific cautions:** Different animals have different optical pulse protocols; the recorded `watts` values have not been physically calibrated; the dataset is not hippocampal epilepsy. Stimulation-locked electrical artifacts and concurrent injected current must be considered.

## Run in Kaggle

1. Upload this notebook as a Kaggle notebook. Set **Internet = ON**, **Accelerator = None** (GPU unnecessary). No Kaggle dataset input is required.
2. Select **Run All**. The first run is capped at **5 mice, ≤125 MB per file, ≤450 MB total**. Default expected download is approximately **402 MB**.
3. The notebook deletes raw downloaded NWB files after analysis. It saves only small CSV/PNG/JSON outputs.
4. Download **`phase2dc_response_verification_results.zip`** in `/kaggle/working/` and upload it to the same ChatGPT project.

To reduce download/time for a first trial, set `MAX_NEW_MICE = 0` (reinspect 3 earlier mice). The recommended full audit uses `MAX_NEW_MICE = 2`.

In [1]:
import os, json, re, traceback, zipfile, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import h5py
import requests
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

DANDI_ID, VERSION = '000005', '0.220126.1853'
API = 'https://api.dandiarchive.org'
OUT = Path('/kaggle/working/phase2dc_outputs') if Path('/kaggle/working').exists() else Path('/mnt/data/phase2dc_outputs')
CACHE = OUT / 'temporary_raw_nwb'
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

PREVIOUS_MICE = ['sub-anm266945', 'sub-anm266951', 'sub-anm281592']
MAX_NEW_MICE = 2                # 2 more independent mice, ~105 and ~114 MB
MAX_PER_FILE_MB = 125.0
MAX_TOTAL_DOWNLOAD_MB = 450.0
MAX_TIMESERIES_SAMPLES = 5_000_000
MAX_PULSES_PER_MOUSE = 3000
MAX_EVENT_PLOTS_PER_MOUSE = 2
MIN_ISOLATION_GAP_S = 0.20
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
DELETE_RAW_NWB_AFTER_PROCESSING = True
ERRORS = []

HTTP = requests.Session()
HTTP.mount('https://', HTTPAdapter(max_retries=Retry(
    total=3, connect=3, read=2, backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504], allowed_methods=['GET'])))
HTTP.headers.update({'User-Agent':'Optogenetic-phase2dc-audit/1.0'})
print('Phase 2D-C | CPU sufficient | Python scientific packages preinstalled')
print('Published DANDI version:', VERSION, '| max total MB:', MAX_TOTAL_DOWNLOAD_MB)

Phase 2D-C | CPU sufficient | Python scientific packages preinstalled
Published DANDI version: 0.220126.1853 | max total MB: 450.0


## 1 | Select independent animals from the published DANDI manifest

The manifest selection is based on **file size and subject IDs**, **not** on an assumption that every optogenetics-marked recording contains useable events. We add the two smallest optogenetics-marked subject files outside the original three.

In [2]:
def get_manifest():
    url = f'{API}/api/dandisets/{DANDI_ID}/versions/{VERSION}/assets/?page_size=1000'
    rows = []
    for _ in range(60):
        res = HTTP.get(url, timeout=(20, 90), headers={'Accept':'application/json'})
        res.raise_for_status()
        body = res.json()
        rows.extend(body.get('results', []))
        url = body.get('next')
        if not url: break
    else:
        raise RuntimeError('DANDI manifest pagination limit hit')
    frame = pd.DataFrame(rows)
    for col in ['asset_id', 'path', 'size']:
        if col not in frame: raise RuntimeError(f'Missing manifest column: {col}')
    frame['size_bytes'] = pd.to_numeric(frame['size'], errors='coerce')
    frame['size_MB'] = frame['size_bytes'] / 1e6
    frame['mouse'] = frame['path'].str.extract(r'(sub-[^/]+)', expand=False)
    frame['candidate_ogen'] = frame['path'].str.contains(r'\+ogen\.nwb$',regex=True,na=False)
    frame['candidate_icephys'] = frame['path'].str.contains(r'\+icephys',regex=True,na=False)
    return frame

manifest = get_manifest()
manifest.to_csv(OUT/'published_manifest.csv',index=False)
eligible = manifest.loc[(manifest.candidate_ogen) & (manifest.candidate_icephys) &
    (manifest.size_MB<=MAX_PER_FILE_MB) & (manifest.size_bytes>0)].copy().sort_values('size_MB')
first = []
for mouse in PREVIOUS_MICE:
    group=eligible[eligible.mouse==mouse]
    if group.empty: print('WARNING: expected mouse missing from size-capped manifest:',mouse)
    else: first.append(group.iloc[0])
additional = eligible[~eligible.mouse.isin(PREVIOUS_MICE)].drop_duplicates('mouse').head(MAX_NEW_MICE)
selection = pd.DataFrame(first + [row for _,row in additional.iterrows()]).drop_duplicates('mouse')
selection['selection_type'] = np.where(selection.mouse.isin(PREVIOUS_MICE),'prior_verified_metadata','new_subject_candidate')
selection.to_csv(OUT/'selected_mouse_files.csv',index=False)
print('Manifest:',len(manifest),'files /',manifest.mouse.nunique(),'subject folders')
print('Selected:',len(selection),'independent subjects; published file sizes:',round(selection.size_MB.sum(),2),'MB')
display(selection[['mouse','size_MB','selection_type','path']])
assert selection.mouse.is_unique

Manifest: 148 files / 55 subject folders
Selected: 5 independent subjects; published file sizes: 401.98 MB


,mouse,size_MB,selection_type,path
88,sub-anm266945,46.543184,prior_verified_metadata,sub-anm266945/sub-anm266945_ses-20141221_obj-1...
92,sub-anm266951,58.123096,prior_verified_metadata,sub-anm266951/sub-anm266951_ses-20141201_behav...
108,sub-anm281592,78.389912,prior_verified_metadata,sub-anm281592/sub-anm281592_ses-20150309_behav...
52,sub-anm244025,104.930672,new_subject_candidate,sub-anm244025/sub-anm244025_ses-20141024_obj-1...
73,sub-anm260073,113.990856,new_subject_candidate,sub-anm260073/sub-anm260073_ses-20141008_behav...


## 2 | Download only the selected files (strict size limits)

The files are downloaded from DANDI's published object store through its API. Downloads are bounded by the selected asset's published size; files over quota are skipped and reported.

In [3]:
def fetch_asset(row):
    expected = int(row['size_bytes'])
    if expected <= 0 or expected > MAX_PER_FILE_MB*1e6:
        raise ValueError('Asset violates file size cap')
    dest = CACHE / (str(row['asset_id'])+'.nwb')
    if dest.exists() and dest.stat().st_size==expected:
        return dest, 0
    part = dest.with_suffix('.part')
    size=0
    try:
        with HTTP.get(f"{API}/api/assets/{row['asset_id']}/download/", stream=True, timeout=(30, 180)) as response:
            response.raise_for_status()
            with open(part,'wb') as f:
                for piece in response.iter_content(chunk_size=1024*1024):
                    if not piece: continue
                    size += len(piece)
                    if size > expected+4096 or size > MAX_PER_FILE_MB*1e6:
                        raise RuntimeError('Transfer exceeded file size safety limit')
                    f.write(piece)
        if size!=expected: raise IOError(f'Transfer size mismatch expected={expected} got={size}')
        part.replace(dest)
        return dest,size
    finally:
        if part.exists():part.unlink()

LOCAL_FILES={}
transferred=0
for _,row in selection.iterrows():
    if transferred+int(row.size_bytes)>MAX_TOTAL_DOWNLOAD_MB*1e6:
        ERRORS.append({'stage':'quota','mouse':row.mouse,'error':'Skipped for total download cap'})
        continue
    try:
        p,n=fetch_asset(row)
        LOCAL_FILES[str(row.asset_id)] = p
        transferred+=n
        print('Downloaded/opened:',row.mouse,'|',round(p.stat().st_size/1e6,2),'MB')
    except Exception as exc:
        ERRORS.append({'stage':'download','mouse':row.mouse,'error':repr(exc)})
        print('Download error:',row.mouse,repr(exc))
print('Available NWB files:',len(LOCAL_FILES),'transferred MB:',round(transferred/1e6,2))

Downloaded/opened: sub-anm266945 | 46.54 MB
Downloaded/opened: sub-anm266951 | 58.12 MB
Downloaded/opened: sub-anm281592 | 78.39 MB
Downloaded/opened: sub-anm244025 | 104.93 MB
Downloaded/opened: sub-anm260073 | 113.99 MB
Available NWB files: 5 transferred MB: 401.98


## 3 | Interpret NWB optical, membrane-voltage and current-injection series

Time axes are read from explicit `timestamps` or NWB `starting_time` + rate; voltages are converted to **mV only when the stored units support that conversion**. Raw optical codes are **not interpreted as calibrated watts**. We retain current-injection metadata because changes in injected current can confound voltage responses.

In [4]:
def as_text(v):
    if isinstance(v,bytes):return v.decode('utf-8','replace')
    if isinstance(v,np.ndarray):
        if v.size==0:return ''
        return as_text(v.reshape(-1)[0])
    return str(v)

def neuro_type(g):
    return as_text(g.attrs.get('neurodata_type', '')).lower()

def find_timeseries(hf):
    out={'opto':[],'voltage':[],'current_injection':[]}
    def visitor(path,obj):
        if not isinstance(obj,h5py.Group) or not isinstance(obj.get('data'),h5py.Dataset):return
        typ=neuro_type(obj)
        pl=path.lower()
        if 'optogeneticseries' in typ:
            out['opto'].append(path)
        elif 'currentclampstimulusseries' in typ:
            out['current_injection'].append(path)
        elif 'currentclampseries' in typ and pl.startswith('acquisition/'):  # VoltageClampSeries records CURRENT, not voltage
            out['voltage'].append(path)
    hf.visititems(visitor)
    return out

def h5_numeric_time_and_data(g, n_cap=MAX_TIMESERIES_SAMPLES):
    ds=g.get('data')
    if not isinstance(ds,h5py.Dataset) or ds.ndim not in (1,2):
        raise ValueError('Time series must have a 1-D or 2-D numeric data array')
    n=int(ds.shape[0])
    if n<3 or n>n_cap:raise ValueError(f'Time series {n} samples outside bounded analysis range')
    if ds.ndim==1: raw=np.asarray(ds[:],dtype=np.float64)
    elif ds.shape[1]==1: raw=np.asarray(ds[:,0],dtype=np.float64)
    else:raise ValueError('Multichannel data: avoid arbitrarily selecting a channel')
    ts=g.get('timestamps')
    if isinstance(ts,h5py.Dataset) and ts.shape==(n,):
        t=np.asarray(ts[:],dtype=np.float64); mode='explicit_timestamps'
    elif isinstance(g.get('starting_time'),h5py.Dataset):
        start=g['starting_time']
        t0=float(np.asarray(start[()]).reshape(-1)[0])
        rate=float(np.asarray(start.attrs.get('rate',np.nan)).reshape(-1)[0])
        if not np.isfinite(rate) or rate<=0:raise ValueError('Invalid NWB rate')
        t=t0+np.arange(n,dtype=np.float64)/rate;mode='starting_time_rate'
    else:
        raise ValueError('No matching NWB timestamps or starting time and rate')
    if not (np.isfinite(t).all() and np.isfinite(raw).all() and np.all(np.diff(t)>0)):
        raise ValueError('Nonfinite data or nonmonotonic time axis')
    attrs=ds.attrs
    conv=float(np.asarray(attrs.get('conversion',1.)).reshape(-1)[0])
    offset=float(np.asarray(attrs.get('offset',0.)).reshape(-1)[0])
    if not np.isfinite(conv) or not np.isfinite(offset):raise ValueError('Invalid NWB scaling')
    physical=raw*conv+offset
    unit=as_text(ds.attrs.get('unit',g.attrs.get('unit','unknown')))
    return t,physical,unit,mode

def to_mV(vals,unit):
    u=unit.strip().lower()
    if u in ('volts','volt','v'):return vals*1000.0
    if u in ('mv','millivolts','millivolt'):return vals
    raise ValueError(f'Cannot convert voltage unit {unit!r} to mV safely')

def get_optical_events(t, vals, base):
    # These are candidate command transitions. Physical illumination is NOT independently confirmed.
    if len(np.unique(vals))>100 or not (np.any(vals==0) and np.any(vals>0)):
        return pd.DataFrame(), 'Optical values not interpretable as positive-versus-zero light commands'
    active=vals>0
    rises=np.flatnonzero((~active[:-1]) & active[1:])+1
    falls=np.flatnonzero(active[:-1] & (~active[1:]))+1
    out=[]
    for a in rises[:MAX_PULSES_PER_MOUSE]:
        j=np.searchsorted(falls,a,side='right')
        if j>=len(falls):continue
        b=int(falls[j]); a=int(a)
        if b<=a:continue
        start,stop=float(t[a]),float(t[b])
        if stop<=start:continue
        out.append({**base,'onset_s':start,'offset_s':stop,
             'duration_ms':1000*(stop-start),'raw_optical_code':float(vals[a])})
    return pd.DataFrame(out),'Candidate positive-zero optical intervals; unit/code not calibrated'

## 4 | Event-level voltage and candidate-spike measurements

For each event we compute: **baseline** (−200 to −20 ms), **early** (+5 to +50 ms), **delayed** (+50 to +150 ms), **after-light-offset** (+10 to +200 ms), baseline-corrected voltage change, and candidate spike counts (if the membrane voltage crosses a conservative, explicitly stated detector). Short 5-ms pulses and 1–1.5-s pulses are analyzed separately.

A pulse is flagged **isolated** only when no other detected optical interval falls within 200 ms before its onset or after its offset. Contaminated or poorly covered windows are not silently treated as clean observations.

In [5]:
def interval_values(times, vals, start, stop):
    if not(np.isfinite(start) and np.isfinite(stop) and stop>start):return np.array([])
    lo=int(np.searchsorted(times,start,side='left'))
    hi=int(np.searchsorted(times,stop,side='left'))
    if hi-lo < 4:return np.array([])
    sub_t=times[lo:hi]
    median_dt=float(np.median(np.diff(sub_t)))
    if median_dt<=0 or float(np.max(np.diff(sub_t)))>max(0.020,10*median_dt):return np.array([])
    if sub_t[0]>start+max(0.01,10*median_dt) or sub_t[-1]<stop-max(0.01,10*median_dt):
        return np.array([])
    return vals[lo:hi]

def group_duration(ms):
    if ms<25:return 'short_lt25ms'
    if ms<250:return 'medium_25to250ms'
    return 'long_ge250ms'

def get_voltage_event_rows(ev, voltage_t, voltage_mv, spike_times=None, current=None):
    all_rows=[]
    ev=ev.sort_values('onset_s').reset_index(drop=True)
    onset=ev.onset_s.to_numpy(dtype=float)
    offset=ev.offset_s.to_numpy(dtype=float)
    for i,r in ev.iterrows():
        a,b=float(r.onset_s),float(r.offset_s)
        prev = float(offset[i-1]) if i>0 else -np.inf
        nxt = float(onset[i+1]) if i+1<len(ev) else np.inf
        isolated = bool((a-prev>=MIN_ISOLATION_GAP_S) and (nxt-b>=MIN_ISOLATION_GAP_S))
        base=interval_values(voltage_t,voltage_mv,a-.20,a-.02)
        early=interval_values(voltage_t,voltage_mv,a+.005,a+.05)
        delayed=interval_values(voltage_t,voltage_mv,a+.05,a+.15)
        after=interval_values(voltage_t,voltage_mv,b+.01,b+.20)
        base_med=float(np.median(base)) if len(base) else np.nan
        e_mean=float(np.mean(early)) if len(early) else np.nan
        d_mean=float(np.mean(delayed)) if len(delayed) else np.nan
        f_mean=float(np.mean(after)) if len(after) else np.nan
        # 'Peak latency' is a descriptive timing feature, not a biophysical response diagnosis.
        dt_window=interval_values(voltage_t,voltage_mv,a+.005,a+.15)
        latency_ms=np.nan
        if len(dt_window) and np.isfinite(base_med):
            ix0=int(np.searchsorted(voltage_t,a+.005))
            ix1=ix0+len(dt_window)
            tt=voltage_t[ix0:ix1]
            if len(tt)==len(dt_window):
                k=int(np.argmax(np.abs(dt_window-base_med)))
                latency_ms=1000*(float(tt[k])-a)
        n_pre=n_early=np.nan
        if spike_times is not None:
            n_pre=int(np.searchsorted(spike_times,a-.02)-np.searchsorted(spike_times,a-.20))
            n_early=int(np.searchsorted(spike_times,a+.05)-np.searchsorted(spike_times,a+.005))
        injection_flag='not_recorded'
        inj_delta=np.nan
        if current is not None:
            it,iv,iu=current
            ci=interval_values(it,iv,a-.20,a-.02)
            cj=interval_values(it,iv,a+.005,a+.05)
            if len(ci) and len(cj):
                inj_delta=float(np.mean(cj)-np.mean(ci))
                noise=float(np.median(np.abs(ci-np.median(ci)))*1.4826)
                threshold=max(noise*5, abs(np.median(ci))*0.01, 1e-15)
                injection_flag='possible_step' if abs(inj_delta)>threshold else 'no_large_step_detected'
            else:
                injection_flag='not_aligned_or_unavailable'
        all_rows.append({**r.to_dict(),'event_index':i,'protocol_group':group_duration(1000*(b-a)),
            'isolated_200ms_each_side':isolated,
            'baseline_mV':base_med,'early_mean_mV':e_mean,'delayed_mean_mV':d_mean,'post_offset_mean_mV':f_mean,
            'early_minus_baseline_mV':e_mean-base_med,'delayed_minus_baseline_mV':d_mean-base_med,
            'post_offset_minus_baseline_mV':f_mean-base_med,
            'descriptive_peak_latency_ms':latency_ms,
            'candidate_spikes_pre':n_pre,'candidate_spikes_early':n_early,
            'candidate_spike_rate_baseline_hz':(n_pre/0.18 if np.isfinite(n_pre) else np.nan),
            'candidate_spike_rate_early_hz':(n_early/0.045 if np.isfinite(n_early) else np.nan),
            'current_injection_change_flag':injection_flag,
            'current_injection_early_pre_delta_raw':inj_delta,
            'time_windows_complete':bool(len(base) and len(early) and len(delayed)),
            'next_opto_onset_gap_s':float(nxt-b),
            'previous_opto_offset_gap_s':float(a-prev)})
    return pd.DataFrame(all_rows)

def candidate_spikes(t,v):
    dt=float(np.median(np.diff(t)))
    fs=1/dt
    if not np.isfinite(fs) or fs<500:
        return np.array([]), 'Not assessed: sampling below 500 Hz'
    peaks,_=find_peaks(v,height=-20,prominence=25,distance=max(1,int(0.002/dt)))
    return t[peaks], 'Candidate AP peaks: voltage > -20 mV, prominence >=25 mV, >=2 ms apart; NOT ground-truth spikes'

REL_TIME=np.arange(-0.2,0.151,0.002)
def aligned_waveform(times,values,onsets):
    waves=[]
    for o in onsets:
        if o+REL_TIME[0]<times[0] or o+REL_TIME[-1]>times[-1]:continue
        xx=o+REL_TIME
        left=np.searchsorted(times,xx[0]);right=np.searchsorted(times,xx[-1],side='right')
        if right-left<4:continue
        local_t=times[left:right]
        if np.max(np.diff(local_t))>max(.02,10*np.median(np.diff(local_t))):continue
        z=np.interp(xx,local_t,values[left:right])
        z-=float(np.median(z[(REL_TIME>=-.20)&(REL_TIME<-.02)]))
        waves.append(z)
    if not waves:return None
    return np.asarray(waves,dtype=float)

def temporal_shift_control(valid_events, all_event_onsets, voltage_t, voltage_mv, repeats=200):
    # Exploratory time-shift negative control; not proof of stimulation causality.
    valid_events=valid_events.dropna(subset=['early_minus_baseline_mV'])
    observed=float(valid_events.early_minus_baseline_mV.mean()) if len(valid_events) else np.nan
    status={'n_original_events':len(valid_events),'observed_mean_delta_mV':observed,
            'test_type':'circular time shift of ON timestamps with exclusion around real ON events',
            'status':'insufficient_trials','n_shift_replicates':0,
            'null_mean_delta_mV':np.nan,'exploratory_two_sided_p':np.nan}
    span=float(voltage_t[-1]-voltage_t[0])
    if len(valid_events)<20 or span<8:return status
    actual=np.sort(np.asarray(all_event_onsets,dtype=float))
    deltas=[]
    for _ in range(repeats):
        shift=float(rng.uniform(2.0,span-2.0))
        onsets=voltage_t[0]+np.mod(valid_events.onset_s.to_numpy(dtype=float)-voltage_t[0]+shift,span)
        # Exclude synthetic times outside complete evaluation window.
        mask=(onsets>voltage_t[0]+.22)&(onsets<voltage_t[-1]-.18)
        onsets=onsets[mask]
        if not len(onsets):continue
        nearest=np.searchsorted(actual,onsets)
        l=actual[np.maximum(nearest-1,0)]
        r=actual[np.minimum(nearest,len(actual)-1)]
        dist=np.minimum(np.abs(onsets-l),np.abs(onsets-r))
        onsets=onsets[dist>=.20]
        effects=[]
        for o in onsets:
            aa=interval_values(voltage_t,voltage_mv,o-.20,o-.02)
            bb=interval_values(voltage_t,voltage_mv,o+.005,o+.05)
            if len(aa) and len(bb): effects.append(float(np.mean(bb)-np.median(aa)))
        if len(effects)>=max(10,len(valid_events)//3):deltas.append(float(np.mean(effects)))
    if len(deltas)<30:
        status['status']='too_few_valid_shift_replicates'
        status['n_shift_replicates']=len(deltas)
        return status
    null=np.asarray(deltas)
    status.update({'status':'exploratory_null_available','n_shift_replicates':len(null),
       'null_mean_delta_mV':float(np.mean(null)),
       'exploratory_two_sided_p':float((1+np.sum(np.abs(null-np.mean(null))>=abs(observed-np.mean(null))))/(len(null)+1))})
    return status

## 5 | Inspect each real mouse, compute event responses and save actual plots

Every mouse gets its own subject-level output. If the metadata are incompatible or a time axis is invalid, that mouse is marked as **failed**, not silently discarded or counted as verified.

In [6]:
all_events=[]
all_rows=[]
series_inventory=[]
subject_rows=[]
mean_traces=[]
SHAM_RESULTS=[]
for _,sel in selection.iterrows():
    mouse=str(sel.mouse)
    path=LOCAL_FILES.get(str(sel.asset_id))
    if path is None: continue
    try:
        with h5py.File(path,'r') as hf:
            sid_obj=hf.get('/general/subject/subject_id')
            sid=as_text(sid_obj[()]) if isinstance(sid_obj,h5py.Dataset) else ''
            if sid != mouse.removeprefix('sub-'):
                raise ValueError(f'Subject ID mismatch {sid} vs {mouse}')
            groups=find_timeseries(hf)
            if not groups['opto'] or not groups['voltage']:
                raise ValueError(f'Missing optical or acquisition voltage series: {groups}')
            vpath=groups['voltage'][0]
            vt, vv, vu, vmode = h5_numeric_time_and_data(hf[vpath])
            vmv=to_mV(vv,vu)
            vdt=float(np.median(np.diff(vt)))
            spike_t, spike_rule = candidate_spikes(vt,vmv)
            inj=None
            for ipath in groups['current_injection'][:1]:
                try:
                    it,iv,iu,imode=h5_numeric_time_and_data(hf[ipath])
                    inj=(it,iv,iu)
                except Exception as exc:
                    ERRORS.append({'stage':'injection_read','mouse':mouse,'error':repr(exc)})
            optical=[]
            for opath in groups['opto']:
                ot,ov,ou,omode=h5_numeric_time_and_data(hf[opath])
                E,note=get_optical_events(ot,ov,{'mouse':mouse,'asset_id':str(sel.asset_id),
                     'file_path':sel.path,'optical_series':opath,'optical_unit_stored':ou})
                series_inventory.append({'mouse':mouse,'kind':'optical','path':opath,'stored_unit':ou,
                     'timing_mode':omode,'n_samples':len(ot),'t_start_s':float(ot[0]),'t_end_s':float(ot[-1]),
                     'sample_period_ms_median':1000*float(np.median(np.diff(ot))), 'events':len(E),
                     'annotation':note})
                if len(E):optical.append(E)
            if not optical:raise ValueError('No complete positive-zero optical intervals')
            ev=pd.concat(optical,ignore_index=True).sort_values('onset_s').reset_index(drop=True)
            # Site overlap can create simultaneous commands. Warn; trial independence is not guaranteed.
            ev['next_onset_ms']=1000*(ev.onset_s.shift(-1)-ev.onset_s)
            ev.to_csv(OUT/f'{mouse}_optical_events.csv',index=False)
            all_events.extend(ev.to_dict('records'))
            measurements=get_voltage_event_rows(ev,vt,vmv,spike_t,current=inj)
            all_rows.extend(measurements.to_dict('records'))
            series_inventory.append({'mouse':mouse,'kind':'membrane_voltage','path':vpath,
                 'stored_unit':vu,'timing_mode':vmode,'n_samples':len(vt),
                 't_start_s':float(vt[0]),'t_end_s':float(vt[-1]),
                 'sample_period_ms_median':1000*vdt,'events':0,
                 'annotation':spike_rule})
            if groups['current_injection']:
                series_inventory.append({'mouse':mouse,'kind':'current_injection','path':groups['current_injection'][0],
                     'stored_unit':inj[2] if inj else 'not_read','timing_mode':'read' if inj else 'unverified',
                     'n_samples':len(inj[0]) if inj else 0,'t_start_s':float(inj[0][0]) if inj else np.nan,
                     't_end_s':float(inj[0][-1]) if inj else np.nan,
                     'sample_period_ms_median':1000*np.median(np.diff(inj[0])) if inj else np.nan,
                     'events':0,'annotation':'Concurrent injection potential confound'})
            valid=measurements.loc[measurements.isolated_200ms_each_side & measurements.time_windows_complete].copy()
            valid=valid[valid.current_injection_change_flag!='possible_step']
            means={}
            # Plot up to two real event examples; high-density recordings are not rendered fully.
            first_times=valid.onset_s.head(MAX_EVENT_PLOTS_PER_MOUSE).tolist()
            if not first_times:first_times=ev.onset_s.head(MAX_EVENT_PLOTS_PER_MOUSE).tolist()
            fig,axes=plt.subplots(len(first_times),1,figsize=(9,3.0*len(first_times)),squeeze=False)
            for ax,o in zip(axes.flat,first_times):
                aa=float(o)-.21;bb=float(o)+.22
                idx0=np.searchsorted(vt,aa);idx1=np.searchsorted(vt,bb)
                if idx1>idx0:
                    ax.plot((vt[idx0:idx1]-o)*1000,vmv[idx0:idx1],linewidth=.85)
                right=float(ev.iloc[np.argmin(np.abs(ev.onset_s-o))].offset_s)
                ax.axvspan(0,min(220.,1000*(right-o)),color='gray',alpha=.16,label='Optical command ON')
                ax.axvline(0,color='black',ls='--',lw=.7)
                ax.set(xlabel='Time from optical ON (ms)',ylabel='Membrane potential (mV)')
                ax.legend(loc='best',fontsize=7)
            fig.suptitle(f'{mouse}: actual NWB voltage around candidate optical ON events')
            fig.tight_layout()
            fig.savefig(OUT/f'{mouse}_event_voltage_examples.png',dpi=170);plt.close(fig)
            # Event-triggered mean from sufficiently isolated and unconfounded trials only.
            if len(valid):
                waves=aligned_waveform(vt,vmv,valid.onset_s.to_numpy())
                if waves is not None:
                    mean=np.mean(waves,axis=0)
                    sem=np.std(waves,axis=0,ddof=1)/np.sqrt(len(waves)) if len(waves)>1 else np.full_like(mean,np.nan)
                    mean_traces.extend([{'mouse':mouse,'relative_ms':float(1000*t),'mean_baseline_corrected_mV':float(v),
                      'sem_mV':float(e),'n_events':len(waves)} for t,v,e in zip(REL_TIME,mean,sem)])
                    fig,ax=plt.subplots(figsize=(9,3.5))
                    ax.plot(REL_TIME*1000,mean,linewidth=1.2,label='Mean voltage minus pre-onset baseline')
                    if len(waves)>1:ax.fill_between(REL_TIME*1000,mean-sem,mean+sem,alpha=.18)
                    ax.axvline(0,ls='--',c='black',lw=.8)
                    ax.set(xlabel='Time from candidate light ON (ms)',ylabel='Baseline-corrected membrane voltage (mV)',
                           title=f'{mouse}: event-locked mean (n={len(waves)} isolated trials)')
                    ax.legend();fig.tight_layout();fig.savefig(OUT/f'{mouse}_event_locked_mean.png',dpi=180);plt.close(fig)
            SHAM_RESULTS.append({'mouse':mouse,**temporal_shift_control(valid,ev.onset_s.to_numpy(),vt,vmv)})
            n_good=int(len(valid))
            subject_rows.append({'mouse':mouse,'status':'recording_extracted','n_candidate_light_events':len(ev),
                'n_isolated_clean_events':n_good,
                'n_complete_voltage_windows':int(measurements.time_windows_complete.sum()),
                'n_detected_candidate_spikes':len(spike_t),'spike_detector_rule':spike_rule,
                'candidate_spike_early_window_probability_clean':float((valid.candidate_spikes_early>0).mean()) if n_good else np.nan,
                'candidate_spike_early_rate_hz_clean':float(valid.candidate_spike_rate_early_hz.mean()) if n_good else np.nan,
                'median_pulse_ms':float(ev.duration_ms.median()),
                'protocol_class':group_duration(float(ev.duration_ms.median())),
                'raw_optical_codes':'|'.join(map(str,sorted(ev.raw_optical_code.unique()))),
                'stored_optical_unit':'|'.join(sorted(ev.optical_unit_stored.unique())),
                'neural_sample_interval_ms':1000*vdt,
                'voltage_unit_original':vu,'voltage_timing_mode':vmode,
                'voltage_t_start_s':float(vt[0]),'voltage_t_end_s':float(vt[-1]),
                'n_current_injection_series':len(groups['current_injection']),
                'n_event_windows_with_possible_injection_steps':int((measurements.current_injection_change_flag=='possible_step').sum()),
                'n_event_windows_with_unresolved_injection':int((measurements.current_injection_change_flag.isin(['not_recorded','not_aligned_or_unavailable'])).sum()),
                'early_minus_baseline_mV_median_clean':float(valid.early_minus_baseline_mV.median()) if n_good else np.nan,
                'delayed_minus_baseline_mV_median_clean':float(valid.delayed_minus_baseline_mV.median()) if n_good else np.nan})
            print(mouse,'events=',len(ev),'isolated/complete/unconfounded=',n_good,'median duration ms=',round(ev.duration_ms.median(),2))
    except Exception as exc:
        ERRORS.append({'stage':'NWB_response_analysis','mouse':mouse,'error':repr(exc)+' | '+traceback.format_exc(limit=2)})
        subject_rows.append({'mouse':mouse,'status':'FAILED','error':repr(exc)})
        print('FAILED:',mouse,repr(exc))
    finally:
        if DELETE_RAW_NWB_AFTER_PROCESSING and path.exists():
            path.unlink()

S=pd.DataFrame(subject_rows)
E=pd.DataFrame(all_events)
R=pd.DataFrame(all_rows)
I=pd.DataFrame(series_inventory)
T=pd.DataFrame(mean_traces)
N=pd.DataFrame(SHAM_RESULTS)
N.to_csv(OUT/'temporal_shift_negative_control_EXPLORATORY.csv',index=False)
S.to_csv(OUT/'mouse_level_feasibility.csv',index=False)
E.to_csv(OUT/'candidate_optical_events_UNCALIBRATED.csv',index=False)
R.to_csv(OUT/'event_level_voltage_responses_EXPLORATORY.csv',index=False)
I.to_csv(OUT/'series_and_timebase_inventory.csv',index=False)
T.to_csv(OUT/'event_locked_average_voltage.csv',index=False)
pd.DataFrame(ERRORS,columns=['stage','mouse','error']).to_csv(OUT/'errors_and_warnings.csv',index=False)
display(S)

sub-anm266945 events= 5 isolated/complete/unconfounded= 5 median duration ms= 1499.9
sub-anm266951 events= 5 isolated/complete/unconfounded= 5 median duration ms= 999.9
sub-anm281592 events= 298 isolated/complete/unconfounded= 0 median duration ms= 5.0
sub-anm244025 events= 46 isolated/complete/unconfounded= 40 median duration ms= 500.0
sub-anm260073 events= 3350 isolated/complete/unconfounded= 0 median duration ms= 5.0


,mouse,status,n_candidate_light_events,n_isolated_clean_events,n_complete_voltage_windows,n_detected_candidate_spikes,spike_detector_rule,candidate_spike_early_window_probability_clean,candidate_spike_early_rate_hz_clean,median_pulse_ms,...,neural_sample_interval_ms,voltage_unit_original,voltage_timing_mode,voltage_t_start_s,voltage_t_end_s,n_current_injection_series,n_event_windows_with_possible_injection_steps,n_event_windows_with_unresolved_injection,early_minus_baseline_mV_median_clean,delayed_minus_baseline_mV_median_clean
0,sub-anm266945,recording_extracted,5,5,5,65,"Candidate AP peaks: voltage > -20 mV, prominen...",0.0,0.0,1499.9,...,0.1,volts,explicit_timestamps,0.0,155.850799,1,0,0,2.575658,-0.408415
1,sub-anm266951,recording_extracted,5,5,5,161,"Candidate AP peaks: voltage > -20 mV, prominen...",0.0,0.0,999.9,...,0.1,volts,explicit_timestamps,0.0,176.146050,1,0,0,-4.539340,-2.273913
2,sub-anm281592,recording_extracted,298,0,298,0,"Candidate AP peaks: voltage > -20 mV, prominen...",NaN,NaN,5.0,...,0.1,volts,explicit_timestamps,0.0,211.780575,1,0,0,NaN,NaN
3,sub-anm244025,recording_extracted,46,40,46,0,"Candidate AP peaks: voltage > -20 mV, prominen...",0.0,0.0,500.0,...,0.1,volts,explicit_timestamps,0.0,325.387953,1,6,0,-0.130359,-0.071589
4,sub-anm260073,recording_extracted,3350,0,3350,2,"Candidate AP peaks: voltage > -20 mV, prominen...",NaN,NaN,5.0,...,0.1,volts,explicit_timestamps,0.0,318.477815,1,240,0,NaN,NaN


## 6 | Quality gate: comparable protocols and remaining confounders

This step **does not declare synchronization proven**. It reports whether independent animals have comparable pulse durations, enough isolated trials, interpretable voltage units, and no obvious co-occurring injection step. Unobserved confounding and optical artifacts remain possible even if plots show event-locked changes.

In [7]:
summary={
  'dataset_id':DANDI_ID,'pinned_version':VERSION,'generated_utc':datetime.now(timezone.utc).isoformat(),
  'number_of_selected_independent_mice':int(selection.mouse.nunique()),
  'number_of_real_files_successfully_analyzed':int((S.status=='recording_extracted').sum()) if 'status' in S else 0,
  'number_of_verified_biophysical_optical_responses':0,
  'no_causality_or_absolute_sync_claim':True,
  'mouse_epilepsy_or_human_data':False,
  'summary_note':'A voltage trace locked to stored optical event timestamps does not independently prove optical causality or absolute synchronization.'
}
if not S.empty and 'n_candidate_light_events' in S:
    ok=S[S.status=='recording_extracted'].copy()
    if not ok.empty:
        summary['total_candidate_light_intervals']=int(ok.n_candidate_light_events.sum())
        summary['total_isolated_unconfounded_event_windows']=int(ok.n_isolated_clean_events.sum())
        summary['mice_with_at_least_20_isolated_events']=int((ok.n_isolated_clean_events>=20).sum())
        summary['mice_with_at_least_20_clean_events_in_same_protocol_class']={
           str(k):int((v.n_isolated_clean_events>=20).sum()) for k,v in ok.groupby('protocol_class')}
        summary['protocol_distribution']={str(k):int(v) for k,v in ok.protocol_class.value_counts().items()}
        summary['conservative_go_for_cross_animal_training']=bool(any(
           (g.n_isolated_clean_events>=20).sum()>=3 for _,g in ok.groupby('protocol_class')))
        if not summary['conservative_go_for_cross_animal_training']:
            summary['next_recommendation']='Expand comparable protocols or select a different dataset before cross-animal DL.'
        else:
            summary['next_recommendation']='Review actual event-locked waveforms and confounders manually; protocol-comparable animals may support a pilot.'
    fig,ax=plt.subplots(figsize=(9,4.3))
    groups=S[S.status=='recording_extracted']
    x=np.arange(len(groups))
    ax.bar(x-0.18,groups.n_candidate_light_events,width=.36,label='All candidate light events')
    ax.bar(x+0.18,groups.n_isolated_clean_events,width=.36,label='Isolated, complete, no flagged injection step')
    ax.set_yscale('symlog',linthresh=5)
    ax.set_xticks(x,groups.mouse.str.replace('sub-','',regex=False),rotation=30,ha='right')
    ax.set_ylabel('Event count (symlog)')
    ax.set_title('Real-data audit: optical events and usable candidate trials')
    ax.legend(fontsize=8)
    fig.tight_layout();fig.savefig(OUT/'fig_trial_quality_by_mouse.png',dpi=180);plt.close(fig)
if not T.empty:
    fig,ax=plt.subplots(figsize=(9,4.5))
    for mouse,g in T.groupby('mouse'):
        ax.plot(g.relative_ms,g.mean_baseline_corrected_mV,label=f'{mouse} (n={int(g.n_events.iloc[0])})',lw=1.2)
    ax.axvline(0,color='black',lw=.8,ls='--')
    ax.set(xlabel='Time from candidate optical ON (ms)',ylabel='Baseline-corrected membrane voltage (mV)',
       title='Event-locked voltage by mouse (exploratory, unmatched protocols)')
    ax.legend(fontsize=8)
    fig.tight_layout();fig.savefig(OUT/'fig_cross_mouse_voltage_traces_EXPLORATORY.png',dpi=180);plt.close(fig)
(OUT/'phase2dc_feasibility_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))

{
  "dataset_id": "000005",
  "pinned_version": "0.220126.1853",
  "generated_utc": "2026-10-12T12:06:28.553935+00:00",
  "number_of_selected_independent_mice": 5,
  "number_of_real_files_successfully_analyzed": 5,
  "number_of_verified_biophysical_optical_responses": 0,
  "no_causality_or_absolute_sync_claim": true,
  "mouse_epilepsy_or_human_data": false,
  "summary_note": "A voltage trace locked to stored optical event timestamps does not independently prove optical causality or absolute synchronization.",
  "total_candidate_light_intervals": 3704,
  "total_isolated_unconfounded_event_windows": 50,
  "mice_with_at_least_20_isolated_events": 1,
  "mice_with_at_least_20_clean_events_in_same_protocol_class": {
    "long_ge250ms": 1,
    "short_lt25ms": 0
  },
  "protocol_distribution": {
    "long_ge250ms": 3,
    "short_lt25ms": 2
  },
  "conservative_go_for_cross_animal_training": false,
  "next_recommendation": "Expand comparable protocols or select a different dataset before cross-

## 7 | Export reproducible audit results (no invented trials)

The ZIP contains all real-data CSVs, plots and the verdict. **No raw NWB files are included.** Only after reviewing the voltage plots and event-confound flags should we choose whether to build a matched-protocol cross-animal prediction experiment.

In [8]:
notes = [
    'Phase 2D-C DANDI 000005 response verification and animal expansion',
    'Published source: https://dandiarchive.org/dandiset/000005/0.220126.1853',
    'Subjects in first run: ' + ', '.join(selection.mouse.tolist()),
    'This notebook does NOT train machine learning.',
    'Positive-to-zero transitions are CANDIDATE optical commands, not verified physical light delivery.',
    'Raw optical code is NOT calibrated optical power.',
    'Voltage is in mV only when correctly converted from supported NWB units.',
    'Candidate AP detector output is NOT spike-sorting ground truth.',
    'A matched timestamp is NOT evidence of causality or absolute clock synchronization.',
    'Mouse somatosensory data cannot establish human hippocampal epilepsy therapy.',
    'Never mix the same mouse into training and testing in future DL experiments.',
    'See errors_and_warnings.csv and subject-level protocol inventory.'
]
(OUT/'READ_ME_FIRST.txt').write_text(chr(10).join(notes))
result_zip = Path('/kaggle/working/phase2dc_response_verification_results.zip') if Path('/kaggle/working').exists() else Path('/mnt/data/phase2dc_response_verification_results.zip')
with zipfile.ZipFile(result_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file() and f.suffix.lower() in ('.csv','.json','.png','.txt'):
            z.write(f,arcname=f.name)
print('READY TO DOWNLOAD:',result_zip,'|',round(result_zip.stat().st_size/1e6,2),'MB')
print('Upload it here so we can decide whether comparable mouse protocols support leave-one-animal-out DL.')

READY TO DOWNLOAD: /kaggle/working/phase2dc_response_verification_results.zip | 1.79 MB
Upload it here so we can decide whether comparable mouse protocols support leave-one-animal-out DL.
